# YOLO Polyp Detection — Cache-based Evaluation
Run inference **once**, cache all results, then analyze freely without re-running the model.

In [ ]:
%matplotlib inline
from ultralytics import YOLO
import torch
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
from pathlib import Path
from PIL import Image
import random

In [ ]:
MODEL_PATH = "/datadrive/polyp/yolo_output/runs/detect/2026_04_01(1)/weights/best.pt"
DATA_YAML  = "/root/Desktop/polyp/yolo_data/od/data.yaml"
VAL_TXT    = "/root/Desktop/polyp/yolo_data/od/val.txt"
RUN_DIR    = Path(MODEL_PATH).parent.parent
CACHE_PATH = RUN_DIR / "val_cache.pkl"
IOU_THR    = 0.5

device = 'mps' if torch.backends.mps.is_available() else ('0' if torch.cuda.is_available() else 'cpu')
val_images = Path(VAL_TXT).read_text().strip().splitlines()
print(f"Device : {device}")
print(f"Val set: {len(val_images)} images")
print(f"Cache  : {CACHE_PATH}")

## Step 1 — Run Inference Once & Save Cache
> **Skip this cell if `val_cache.pkl` already exists.**

In [ ]:
if CACHE_PATH.exists():
    print(f"Cache already exists at {CACHE_PATH}, skipping inference.")
else:
    model = YOLO(MODEL_PATH)
    print(f"Model loaded. Running inference on {len(val_images)} images ...")

    def load_gt_boxes(img_path):
        label_path = Path(img_path.replace("/images/", "/labels/")).with_suffix(".txt")
        boxes = []
        if label_path.exists() and label_path.stat().st_size > 0:
            w, h = Image.open(img_path).size
            for line in label_path.read_text().strip().splitlines():
                _, cx, cy, bw, bh = map(float, line.split())
                boxes.append((
                    (cx - bw/2)*w, (cy - bh/2)*h,
                    (cx + bw/2)*w, (cy + bh/2)*h,
                ))
        return boxes

    cache = {}
    for idx, img_path in enumerate(val_images):
        res  = model.predict(img_path, conf=0.01, iou=0.5, verbose=False, device=device)
        pred_boxes = res[0].boxes.xyxy.cpu().numpy().tolist() if res[0].boxes else []
        pred_confs = res[0].boxes.conf.cpu().numpy().tolist() if res[0].boxes else []
        gt_boxes   = load_gt_boxes(img_path)
        cache[img_path] = {
            "gt_boxes"  : gt_boxes,
            "pred_boxes": pred_boxes,
            "pred_confs": pred_confs,
            "is_polyp"  : len(gt_boxes) > 0,
        }
        if (idx + 1) % 500 == 0:
            print(f"  {idx+1}/{len(val_images)} done")

    with open(CACHE_PATH, "wb") as f:
        pickle.dump(cache, f)
    print(f"\nCache saved → {CACHE_PATH}  ({CACHE_PATH.stat().st_size/1e6:.1f} MB)")

## Step 2 — Load Cache

In [ ]:
with open(CACHE_PATH, "rb") as f:
    cache = pickle.load(f)

polyp_keys  = [k for k, v in cache.items() if v["is_polyp"]]
normal_keys = [k for k, v in cache.items() if not v["is_polyp"]]
print(f"Cache loaded: {len(cache)} images")
print(f"  Polyp : {len(polyp_keys)}")
print(f"  Normal: {len(normal_keys)}")

## Step 3 — Helper Functions

In [ ]:
def iou_box(b1, b2):
    xi1 = max(b1[0], b2[0]); yi1 = max(b1[1], b2[1])
    xi2 = min(b1[2], b2[2]); yi2 = min(b1[3], b2[3])
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    a1 = (b1[2]-b1[0]) * (b1[3]-b1[1])
    a2 = (b2[2]-b2[0]) * (b2[3]-b2[1])
    return inter / (a1 + a2 - inter + 1e-9)


def eval_at_conf(conf_thr, keys=None):
    """Compute TP/FP/FN from cache at a given confidence threshold.
    Returns dict with tp, fp, fn, precision, recall, f1,
    and per-image lists: fp_cases, fn_cases.
    """
    if keys is None:
        keys = list(cache.keys())

    tp = fp = fn = 0
    img_tp = img_fn = 0
    fp_cases = []
    fn_cases = []

    for k in keys:
        entry  = cache[k]
        gt     = entry["gt_boxes"]
        # filter predictions by conf threshold
        preds  = [b for b, c in zip(entry["pred_boxes"], entry["pred_confs"]) if c >= conf_thr]
        confs  = [c for c in entry["pred_confs"] if c >= conf_thr]

        matched_gt = set()
        img_tp_count = 0
        for p in preds:
            best_iou, best_j = 0, -1
            for j, g in enumerate(gt):
                v = iou_box(p, g)
                if v > best_iou:
                    best_iou, best_j = v, j
            if best_iou >= IOU_THR and best_j not in matched_gt:
                tp += 1; img_tp_count += 1; matched_gt.add(best_j)
            else:
                fp += 1
        fn_img = len(gt) - len(matched_gt)
        fn += fn_img

        if fp > 0 or fn_img > 0:
            if fp > 0:
                fp_cases.append((k, gt, preds, confs))
            if fn_img > 0:
                fn_cases.append((k, gt, preds, confs))

        if entry["is_polyp"]:
            if img_tp_count > 0:
                img_tp += 1
            else:
                img_fn += 1

    prec = tp / (tp + fp + 1e-9)
    rec  = tp / (tp + fn + 1e-9)
    f1   = 2 * prec * rec / (prec + rec + 1e-9)
    return dict(tp=tp, fp=fp, fn=fn,
                precision=prec, recall=rec, f1=f1,
                img_tp=img_tp, img_fn=img_fn,
                fp_cases=fp_cases, fn_cases=fn_cases)

## Step 4 — Confidence Threshold Sweep (no inference)

In [ ]:
thresholds = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.5, 0.6, 0.7]
rows = []
for conf in thresholds:
    r = eval_at_conf(conf)
    rows.append({"conf": conf, "precision": r["precision"],
                 "recall": r["recall"], "f1": r["f1"],
                 "tp": r["tp"], "fp": r["fp"], "fn": r["fn"]})
    print(f"conf={conf:.2f}  P={r['precision']:.3f}  R={r['recall']:.3f}  F1={r['f1']:.3f}  TP={r['tp']}  FP={r['fp']}  FN={r['fn']}")

df_thr = pd.DataFrame(rows)
best_row = df_thr.loc[df_thr["f1"].idxmax()]
CONF_EVAL = float(best_row["conf"])
print(f"\nBest F1 @ conf={CONF_EVAL:.2f}  F1={best_row['f1']:.4f}")
df_thr

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(df_thr["conf"], df_thr["precision"], 'b-o', label="Precision")
axes[0].plot(df_thr["conf"], df_thr["recall"],    'r-o', label="Recall")
axes[0].plot(df_thr["conf"], df_thr["f1"],        'g-o', label="F1")
axes[0].axvline(CONF_EVAL, color='gray', linestyle='--', label=f"Best conf={CONF_EVAL}")
axes[0].set_xlabel("Confidence Threshold")
axes[0].set_title("P / R / F1 vs Confidence")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].fill_between(df_thr["recall"], df_thr["precision"], alpha=0.15, color='blue')
axes[1].plot(df_thr["recall"], df_thr["precision"], 'b-o')
for _, r in df_thr.iterrows():
    axes[1].annotate(f"{r['conf']:.2f}", (r['recall'], r['precision']),
                     textcoords="offset points", xytext=(4, 4), fontsize=7)
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("PR Curve (Confidence Sweep)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("pr_curve.png", dpi=150)
plt.show()

## Step 5 — Metrics at Best Conf

In [ ]:
res_all = eval_at_conf(CONF_EVAL)

print("=" * 50)
print(f"Overall Metrics (conf={CONF_EVAL:.2f}, IoU>={IOU_THR})")
print("=" * 50)
print(f"TP={res_all['tp']}  FP={res_all['fp']}  FN={res_all['fn']}")
print(f"Precision : {res_all['precision']:.4f}")
print(f"Recall    : {res_all['recall']:.4f}")
print(f"F1        : {res_all['f1']:.4f}")

## Step 6 — Training Curves

In [ ]:
df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Training Curves", fontsize=14)

plot_cols = [
    ("train/box_loss", "val/box_loss",  "Box Loss"),
    ("train/cls_loss", "val/cls_loss",  "Cls Loss"),
    ("train/dfl_loss", "val/dfl_loss",  "DFL Loss"),
    ("metrics/precision(B)", None,                    "Precision"),
    ("metrics/recall(B)",    None,                    "Recall"),
    ("metrics/mAP50(B)",     "metrics/mAP50-95(B)",  "mAP"),
]

for ax, (col1, col2, title) in zip(axes.flat, plot_cols):
    if col1 in df.columns:
        ax.plot(df["epoch"], df[col1], label=col1.split("/")[-1])
    if col2 and col2 in df.columns:
        ax.plot(df["epoch"], df[col2], label=col2.split("/")[-1], linestyle='--')
    ax.set_title(title); ax.set_xlabel("Epoch")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150)
plt.show()

## Step 7 — Metrics Bar Chart & Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Metrics bar
labels_ = ["Precision", "Recall", "F1"]
vals_   = [res_all["precision"], res_all["recall"], res_all["f1"]]
bars = axes[0].bar(labels_, vals_, color=['#4C72B0','#DD8452','#55A868'], edgecolor='white')
axes[0].bar_label(bars, fmt='%.3f', padding=3)
axes[0].set_ylim(0, 1.15); axes[0].set_title("Model Performance Metrics")
axes[0].grid(axis='y', alpha=0.3)

# Confusion matrix
cm = np.array([[res_all['tp'], res_all['fn']],
               [res_all['fp'], 0]])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred Pos', 'Pred Neg'],
            yticklabels=['GT Pos',  'GT Neg'],
            ax=axes[1])
axes[1].set_title(f"Confusion Matrix (conf={CONF_EVAL:.2f}, IoU>={IOU_THR})")
axes[1].set_ylabel("Ground Truth"); axes[1].set_xlabel("Prediction")

plt.tight_layout()
plt.savefig("metrics_and_cm.png", dpi=150)
plt.show()

## Step 8 — Sample Predictions Visualization

In [ ]:
N_SHOW = 12
sample_keys = random.sample(list(cache.keys()), N_SHOW)

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
fig.suptitle(f"Predictions (conf={CONF_EVAL:.2f})  Blue=GT  Red=Pred", fontsize=13)

for ax, k in zip(axes.flat, sample_keys):
    entry = cache[k]
    gt    = entry["gt_boxes"]
    preds = [b for b, c in zip(entry["pred_boxes"], entry["pred_confs"]) if c >= CONF_EVAL]
    confs = [c for c in entry["pred_confs"] if c >= CONF_EVAL]

    ax.imshow(np.array(Image.open(k).convert("RGB")))
    for (x1,y1,x2,y2) in gt:
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor='#3399FF', facecolor='none'))
    for (x1,y1,x2,y2), c in zip(preds, confs):
        ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                     linewidth=2, edgecolor='#FF4444', facecolor='none'))
        ax.text(x1, max(y1-4,0), f"{c:.2f}", color='#FF4444', fontsize=7, fontweight='bold',
                bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
    ax.set_title(f"{Path(k).stem[:20]}\nGT={len(gt)} Pred={len(preds)}", fontsize=7)
    ax.axis('off')

plt.tight_layout()
plt.savefig("predictions_vis.png", dpi=150)
plt.show()

## Step 9 — False Positive Cases

In [ ]:
fp_cases = res_all["fp_cases"]
N_FP = min(8, len(fp_cases))
if N_FP == 0:
    print("No FP cases!")
else:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    fig.suptitle("False Positive Cases (Over-detection)  Blue=GT  Green=TP  Red=FP",
                 fontsize=11, color='darkorange')
    for ax, (k, gt, preds, confs) in zip(axes.flat, random.sample(fp_cases, N_FP)):
        ax.imshow(np.array(Image.open(k).convert("RGB")))
        for (x1,y1,x2,y2) in gt:
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='#3399FF', facecolor='none'))
        for (x1,y1,x2,y2), c in zip(preds, confs):
            color = '#00CC44' if any(iou_box((x1,y1,x2,y2), g) >= IOU_THR for g in gt) else '#FF4444'
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor=color, facecolor='none'))
            ax.text(x1, max(y1-4,0), f"{c:.2f}", color=color, fontsize=7, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
        ax.set_title(f"{Path(k).stem[:18]}\nGT={len(gt)} Pred={len(preds)}", fontsize=7)
        ax.axis('off')
    for ax in list(axes.flat)[N_FP:]: ax.axis('off')
    plt.tight_layout()
    plt.savefig("fp_cases.png", dpi=150)
    plt.show()

## Step 10 — False Negative Cases

In [ ]:
fn_cases = res_all["fn_cases"]
N_FN = min(8, len(fn_cases))
if N_FN == 0:
    print("No FN cases!")
else:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    fig.suptitle("False Negative Cases (Missed detections)  Blue=Detected GT  Orange=Missed GT  Red=Pred",
                 fontsize=11, color='red')
    for ax, (k, gt, preds, confs) in zip(axes.flat, random.sample(fn_cases, N_FN)):
        ax.imshow(np.array(Image.open(k).convert("RGB")))
        matched_gt = set()
        for p in preds:
            for j, g in enumerate(gt):
                if iou_box(p, g) >= IOU_THR and j not in matched_gt:
                    matched_gt.add(j)
        for j, (x1,y1,x2,y2) in enumerate(gt):
            color = '#3399FF' if j in matched_gt else '#FF8800'
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2.5, edgecolor=color, facecolor='none'))
            if j not in matched_gt:
                ax.text((x1+x2)/2, (y1+y2)/2, "MISS", ha='center', va='center',
                        color='#FF8800', fontsize=8, fontweight='bold',
                        bbox=dict(facecolor='white', alpha=0.6, pad=1, edgecolor='none'))
        for (x1,y1,x2,y2), c in zip(preds, confs):
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='#FF4444', facecolor='none'))
            ax.text(x1, max(y1-4,0), f"{c:.2f}", color='#FF4444', fontsize=7, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
        ax.set_title(f"{Path(k).stem[:18]}\nGT={len(gt)} Pred={len(preds)}", fontsize=7)
        ax.axis('off')
    for ax in list(axes.flat)[N_FN:]: ax.axis('off')
    plt.tight_layout()
    plt.savefig("fn_cases.png", dpi=150)
    plt.show()

## Step 11 — Normal vs Polyp Performance

In [ ]:
# Polyp images — box-level + image-level
res_polyp = eval_at_conf(CONF_EVAL, keys=polyp_keys)
p_prec = res_polyp["precision"]
p_rec  = res_polyp["recall"]
p_f1   = res_polyp["f1"]
img_sensitivity = res_polyp["img_tp"] / (res_polyp["img_tp"] + res_polyp["img_fn"] + 1e-9)

# Normal images — false alarm rate
n_fp_img = n_tn = 0
for k in normal_keys:
    entry = cache[k]
    has_pred = any(c >= CONF_EVAL for c in entry["pred_confs"])
    if has_pred: n_fp_img += 1
    else:        n_tn     += 1
specificity = n_tn / (n_tn + n_fp_img + 1e-9)
far         = n_fp_img / len(normal_keys)

print("=" * 55)
print(f"[ Polyp Images ]  n={len(polyp_keys)}")
print("=" * 55)
print(f"  Box-level   TP={res_polyp['tp']}  FP={res_polyp['fp']}  FN={res_polyp['fn']}")
print(f"  Precision   : {p_prec:.4f}")
print(f"  Recall      : {p_rec:.4f}")
print(f"  F1          : {p_f1:.4f}")
print(f"  Sensitivity (image-level): {img_sensitivity:.4f}")
print(f"    Detected {res_polyp['img_tp']} / {len(polyp_keys)} polyp images")
print()
print("=" * 55)
print(f"[ Normal Images ]  n={len(normal_keys)}")
print("=" * 55)
print(f"  TN={n_tn}  FP(false alarm)={n_fp_img}")
print(f"  Specificity       : {specificity:.4f}")
print(f"  False Alarm Rate  : {far:.4f}  ({n_fp_img}/{len(normal_keys)} flagged)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Normal vs Polyp Performance", fontsize=13)

# Polyp box-level
bars = axes[0].bar(["Precision","Recall","F1"], [p_prec, p_rec, p_f1],
                   color=['#4C72B0','#DD8452','#55A868'], edgecolor='white')
axes[0].bar_label(bars, fmt='%.3f', padding=3)
axes[0].set_ylim(0, 1.15)
axes[0].set_title(f"Polyp Box-level (n={len(polyp_keys)})")
axes[0].grid(axis='y', alpha=0.3)

# Sensitivity / Specificity
bars2 = axes[1].bar(["Sensitivity\n(Polyp image-level)", "Specificity\n(Normal)"],
                    [img_sensitivity, specificity],
                    color=['#55A868','#4C72B0'], edgecolor='white')
axes[1].bar_label(bars2, fmt='%.3f', padding=3)
axes[1].set_ylim(0, 1.15)
axes[1].set_title("Image-level Detection Rate")
axes[1].grid(axis='y', alpha=0.3)

# Image-level confusion matrix
cm_img = np.array([[res_polyp['img_tp'], res_polyp['img_fn']],
                   [n_fp_img,            n_tn              ]])
sns.heatmap(cm_img, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Polyp','Pred: Normal'],
            yticklabels=['GT: Polyp', 'GT: Normal'],
            ax=axes[2])
axes[2].set_title(f"Image-level Confusion Matrix\n(conf={CONF_EVAL:.2f})")

plt.tight_layout()
plt.savefig("normal_vs_polyp.png", dpi=150)
plt.show()

## Step 12 — False Alarms on Normal Images

In [ ]:
normal_fp_cases = [(k, [b for b,c in zip(cache[k]['pred_boxes'], cache[k]['pred_confs']) if c >= CONF_EVAL],
                       [c for c in cache[k]['pred_confs'] if c >= CONF_EVAL])
                   for k in normal_keys
                   if any(c >= CONF_EVAL for c in cache[k]['pred_confs'])]

N_SHOW_FP = min(8, len(normal_fp_cases))
if N_SHOW_FP == 0:
    print("No false alarms! Perfect on normal images.")
else:
    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    fig.suptitle(f"False Alarms on Normal Images: {len(normal_fp_cases)}  Red=False Alarm Box",
                 fontsize=11, color='red')
    for ax, (k, preds, confs) in zip(axes.flat, random.sample(normal_fp_cases, N_SHOW_FP)):
        ax.imshow(np.array(Image.open(k).convert("RGB")))
        for (x1,y1,x2,y2), c in zip(preds, confs):
            ax.add_patch(patches.Rectangle((x1,y1), x2-x1, y2-y1,
                         linewidth=2, edgecolor='#FF4444', facecolor='none'))
            ax.text(x1, max(y1-4,0), f"{c:.2f}", color='#FF4444', fontsize=7, fontweight='bold',
                    bbox=dict(facecolor='white', alpha=0.5, pad=1, edgecolor='none'))
        ax.set_title(f"{Path(k).stem[:18]}\nPred={len(preds)} box(es)", fontsize=7)
        ax.axis('off')
    for ax in list(axes.flat)[N_SHOW_FP:]: ax.axis('off')
    plt.tight_layout()
    plt.savefig("normal_false_alarm.png", dpi=150)
    plt.show()

## Step 13 — GT Box Size Distribution (TP vs FN)

In [ ]:
hit_areas = []
miss_areas = []

for k in polyp_keys:
    entry = cache[k]
    gt    = entry["gt_boxes"]
    preds = [b for b,c in zip(entry["pred_boxes"], entry["pred_confs"]) if c >= CONF_EVAL]
    for g in gt:
        area    = (g[2]-g[0]) * (g[3]-g[1])
        matched = any(iou_box(p, g) >= IOU_THR for p in preds)
        (hit_areas if matched else miss_areas).append(area)

all_areas = hit_areas + miss_areas
bins = np.linspace(0, np.percentile(all_areas, 98) if all_areas else 1, 40)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].hist(hit_areas,  bins=bins, alpha=0.6, color='green', label=f"TP ({len(hit_areas)})")
axes[0].hist(miss_areas, bins=bins, alpha=0.6, color='red',   label=f"FN ({len(miss_areas)})")
axes[0].set_xlabel("Box Area (pixels²)"); axes[0].set_ylabel("Count")
axes[0].set_title("GT Box Area Distribution (TP vs FN)")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

for arr, label, color in [(hit_areas,"TP",'green'), (miss_areas,"FN",'red')]:
    if arr:
        sa = np.sort(arr)
        axes[1].plot(sa, np.arange(1,len(sa)+1)/len(sa), label=label, color=color)
axes[1].set_xlabel("Box Area (pixels²)"); axes[1].set_ylabel("CDF")
axes[1].set_title("Box Area CDF (TP vs FN)")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("box_size_dist.png", dpi=150)
plt.show()

## Step 14 — Summary Table

In [ ]:
summary = pd.DataFrame([
    {"Group": "Overall (box-level)",      "n": len(cache),       "Precision": round(res_all['precision'],4), "Recall/Sensitivity": round(res_all['recall'],4),    "F1": round(res_all['f1'],4),   "Specificity": "-",                    "False Alarm Rate": "-"},
    {"Group": "Polyp (box-level)",        "n": len(polyp_keys),  "Precision": round(p_prec,4),              "Recall/Sensitivity": round(p_rec,4),               "F1": round(p_f1,4),            "Specificity": "-",                    "False Alarm Rate": "-"},
    {"Group": "Polyp (image-level)",      "n": len(polyp_keys),  "Precision": "-",                          "Recall/Sensitivity": round(img_sensitivity,4),     "F1": "-",                      "Specificity": "-",                    "False Alarm Rate": "-"},
    {"Group": "Normal (image-level)",     "n": len(normal_keys), "Precision": "-",                          "Recall/Sensitivity": "-",                          "F1": "-",                      "Specificity": round(specificity,4),   "False Alarm Rate": round(far,4)},
])
display(summary)
summary.to_csv("eval_summary.csv", index=False)
print("Saved eval_summary.csv")